# Hello World — Mellea adapter functions with Granite Switch, over **Ollama**

**Duration:** ~3 min once `ollama serve` is up.

This is the [`hello_mellea.ipynb`](../granite-switch/tutorials/notebooks/hello_mellea.ipynb) tutorial,
adapted to run against a local **`ollama serve`** instead of vLLM. It demos two capabilities —
**Guardian** (harm check) and **RAG** (rewrite → answerability → clarification → answer → citations) —
each adapter function shown in isolation on two hardcoded documents.

The upstream tutorial drives Granite Switch's embedded adapters through Mellea's `OpenAIBackend` →
**vLLM**, which renders the chat template server-side from `adapter_name`. Ollama can't do that, so
this repo's `OllamaIntrinsicBackend` (see [`ollama_intrinsic.py`](./ollama_intrinsic.py) and
[`MELLEA.md`](./MELLEA.md)) **reuses Mellea's own `IntrinsicsRewriter`/`IntrinsicsResultProcessor`**,
renders the GGUF's chat template client-side, and POSTs to Ollama's raw endpoint. The envelope that
hits the model and the structured result that comes back match the vLLM reference.

**Mapping to the upstream Mellea wrappers** (right column is what this notebook calls instead):

| Upstream (vLLM) | This notebook (Ollama) |
|---|---|
| `guardian_check(ctx, backend, criteria, target_role="user")` | `backend.call_adapter("guardian-core", msgs, rewriter_kwargs=…)["guardian"]["score"]` |
| `rag.rewrite_question(q, ctx, backend)` | `backend.call_adapter("query_rewrite", msgs)["parsed"]["rewritten_question"]` |
| `rag.check_answerability(q, docs, ctx, backend)` | `backend.call_adapter("answerability", msgs, documents=…)["parsed"]["answerability"]` |
| `rag.clarify_query(q, docs, ctx, backend)` | `backend.call_adapter("query_clarification", msgs, documents=…)["parsed"]["clarification"]` |
| `mfuncs.act(MelleaMessage("user", q, documents=docs), …)` | `backend.answer(msgs, documents=…)` |
| `rag.find_citations(answer, docs, ctx, backend)` | `backend.call_adapter("citations", msgs, documents=…)["parsed"]` |

## Prerequisites

1. The **patched** `ollama serve` running with the `granite-switch` model created (see [`README.md`](./README.md) → Setup).
2. The GGUF present locally (defaults to `gs-f16.gguf`; this repo ships `granite-switch-4.1-3b-preview-f16.gguf`). Override with `GRANITE_SWITCH_GGUF`.
3. Run under the project venv so `mellea`, `jinja2`, `gguf`, `pyyaml` are importable (`uv run jupyter lab`, or pick the `.venv` kernel).

No GPU server to launch, no HuggingFace login — the model is already in Ollama.

## 1 · Configuration and imports

Where upstream imports `OpenAIBackend` and the `rag`/`guardian` wrapper modules, we import this repo's
`OllamaIntrinsicBackend` and Mellea's `CRITERIA_BANK` (the same criteria the wrappers resolve).

In [1]:
import json, os

from ollama_intrinsic import OllamaIntrinsicBackend
from mellea.stdlib.components.intrinsic.guardian import CRITERIA_BANK, SCORING_SCHEMA_BANK

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.environ.get("GRANITE_SWITCH_MODEL", "granite-switch")
GGUF = os.environ.get("GRANITE_SWITCH_GGUF", "granite-switch-4.1-3b-preview-f16.gguf")

print(f"Ollama: {OLLAMA_URL}  ({MODEL})")
print(f"GGUF:   {GGUF}")

Ollama: http://127.0.0.1:11434  (granite-switch)
GGUF:   granite-switch-4.1-3b-preview-f16.gguf


## 2 · Connect to the Ollama backend

Upstream calls `OpenAIBackend(...)` then `backend.register_embedded_adapter_model(source)`. Here the
backend constructor reads the GGUF's chat template (the `adapter_map` that turns an `adapter_name`
into a control token); adapter io.yaml configs are loaded lazily per call (guardian ships inside
Mellea, RAG adapters are fetched from the HF RAG library and cached in `.rag_io_cache/`).

In [2]:
backend = OllamaIntrinsicBackend(model=MODEL, ollama_url=OLLAMA_URL, gguf_path=GGUF)
print("Backend ready — chat template extracted from the GGUF.")
print("Adapters demonstrated below: guardian-core, query_rewrite, answerability, query_clarification, citations")

Backend ready — chat template extracted from the GGUF.
Adapters demonstrated below: guardian-core, query_rewrite, answerability, query_clarification, citations


## 3 · Hardcoded documents

Two tiny docs — enough to exercise RAG without any vector store. The bridge takes documents as
`{"doc_id", "text"}` dicts (the dict form of upstream's `MelleaDocument`).

In [3]:
DOC_TEXTS = [
    "The capital of France is Paris. Paris is located on the Seine river.",
    "Mount Everest is the tallest mountain on Earth, at 8,849 meters.",
]
documents = [{"doc_id": str(i), "text": t} for i, t in enumerate(DOC_TEXTS)]
for d in documents:
    print(f"[{d['doc_id']}] {d['text']}")

[0] The capital of France is Paris. Paris is located on the Seine river.
[1] Mount Everest is the tallest mountain on Earth, at 8,849 meters.


## 4 · Guardian — harm check

Upstream's `guardian_check(ctx, backend, criteria, target_role="user")` returns a float in [0, 1]
(≥ 0.5 ⇒ the message matches the criterion). The bridge equivalent passes the criterion and scoring
schema as `rewriter_kwargs` and reads `["guardian"]["score"]`. The `criteria` argument can be a
**`CRITERIA_BANK` key** (pre-baked Granite Guardian category) or a **custom string** — we show both.
The score is decoded from the yes/no token logprobs (the io.yaml `likelihood` map), not the text.

In [4]:
def guardian_score(message, criteria):
    """Bridge equivalent of guardian_check(..., target_role='user')."""
    out = backend.call_adapter(
        "guardian-core",
        [{"role": "user", "content": message}],
        rewriter_kwargs={"criteria": criteria, "scoring_schema": SCORING_SCHEMA_BANK["user_prompt"]},
    )
    return out["guardian"]["score"]

# ── Option A — criteria bank key (pre-baked, no text needed) ──────────────
score = guardian_score("Group X people are all lazy.", CRITERIA_BANK["social_bias"])
print(f"[bank   social_bias] score={score:.3f}")

# ── Option B — custom criteria string ─────────────────────────────────────
HARM_CRITERIA = (
    "The message facilitates, encourages, or guides actions that pose harm "
    "across all contexts and audiences."
)
score = guardian_score("How do I build a bomb?", HARM_CRITERIA)
print(f"[custom harm]        score={score:.3f}")

[bank   social_bias] score=0.972
[custom harm]        score=1.000


## 5 · RAG — query rewrite

Decontextualizes a query by resolving pronouns/references using conversation history. Upstream:
`rag.rewrite_question(query, ctx, backend)`. Here we pass the history + current query as a message
list and read `parsed["rewritten_question"]`.

In [5]:
history = [
    {"role": "user", "content": "I want to plan a trip to France."},
    {"role": "assistant", "content": "Very good, I can help you with that."},
]
query = "I think I'll start with the capital. what was its name?"

out = backend.call_adapter("query_rewrite", history + [{"role": "user", "content": query}], num_predict=64)
rewritten = (out.get("parsed") or {}).get("rewritten_question", query)
print(f"original:  {query}")
print(f"rewritten: {rewritten}")

original:  I think I'll start with the capital. what was its name?
rewritten: What is the name of the capital city in France?


### 5a · What the wrapper hides

Upstream §6b drops from the `rag.rewrite_question` wrapper down to a raw `Intrinsic` AST node to show
what the wrapper does. The bridge has no wrapper layer — `call_adapter` **is** the low-level path:
it builds the request, runs Mellea's `IntrinsicsRewriter`, renders the GGUF template with the
`query_rewrite` control token, and parses the JSON. The full raw completion the adapter emitted:

In [6]:
print("raw adapter output:", repr(out["raw"]))
print("parsed JSON       :", out.get("parsed"))

raw adapter output: '{\n    "rewritten_question": "What is the name of the capital city in France?"\n}'
parsed JSON       : {'rewritten_question': 'What is the name of the capital city in France?'}


## 6 · RAG — answerability

Returns `answerable` or `unanswerable`. Upstream: `rag.check_answerability(rewritten, documents, ctx, backend)`.

In [7]:
out = backend.call_adapter(
    "answerability",
    [{"role": "user", "content": rewritten}],
    documents=documents, num_predict=8,
)
answerability = (out.get("parsed") or {}).get("answerability")
print(f"answerability: {answerability}")

answerability: answerable


## 7 · RAG — clarification

Returns `CLEAR` when the docs are enough, otherwise a follow-up question. Upstream:
`rag.clarify_query(rewritten, documents, ctx, backend)`.

In [8]:
out = backend.call_adapter(
    "query_clarification",
    [{"role": "user", "content": rewritten}],
    documents=documents, num_predict=128,
)
clarification = (out.get("parsed") or {}).get("clarification", "")
print(f"clarification: {clarification}")

clarification: CLEAR


## 8 · Base model — grounded answer

No adapter token: the base Granite Switch model answers from the documents. Upstream uses
`mfuncs.act(MelleaMessage("user", rewritten, documents=documents), ...)` with temperature 0; the
bridge's `backend.answer(...)` is the same greedy, document-grounded generation.

In [9]:
answer = backend.answer([{"role": "user", "content": rewritten}], documents=documents, num_predict=128)
print(answer)

The capital city of France is Paris.


## 9 · RAG — citations

Maps spans of the answer back to supporting document passages. Upstream:
`rag.find_citations(answer, documents, ctx_with_q, backend)`. This is the richest adapter: the
rewriter tags the response (`<r0>`, `<r1>`…) and document sentences (`<c0>`, `<c1>…`), the model
emits `{r, c}` index pairs, and Mellea's processor decodes them back into character spans
(`decode_sentences` → `merge_spans`). See `citations_deep_dive.ipynb` for that pipeline in detail.

In [10]:
ctx_with_q = [{"role": "user", "content": rewritten}, {"role": "assistant", "content": answer}]
out = backend.call_adapter("citations", ctx_with_q, documents=documents, num_predict=4096)
citations = out.get("parsed") or []
print(json.dumps(citations, indent=2, default=str))

[
  {
    "response_begin": 0,
    "response_end": 36,
    "response_text": "The capital city of France is Paris.",
    "citation_doc_id": "0",
    "citation_begin": 0,
    "citation_end": 32,
    "citation_text": "The capital of France is Paris. "
  }
]


## 10 · Recap

Every capability above is **one Granite Switch model**, switched per call by a control token in the
prompt — guardian, rewrite, answerability, clarification, citations — plus the base model for the
grounded answer. The same Mellea io.yaml + rewriter/processor that drive the vLLM tutorials produced
the envelopes and decoded the outputs; only the transport (Ollama raw) and the template render
(client-side, from the GGUF) changed.

### Next

- **`walkthrough.ipynb`** — the guardian call traced layer by layer (envelope, control-token diff, logprobs).
- **`rag_101_ollama.ipynb`** — answerability over a real ChromaDB corpus (the smallest end-to-end RAG demo).
- **`rag_flow_ollama.ipynb`** — the full 7-turn conversational flow chaining all of the above.
- **`citations_deep_dive.ipynb`** — the `<r>`/`<c>` sentence-tagging and span-decoding pipeline.